In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.image
import matplotlib.pyplot as plt
import numpy as np
from mpltools import annotation

# Session 2: Further Differentiable Programming with Autograd and JAX

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Set up codespace now. (It will take a while!)</b>

We will show you where to find this notebook in the repo while you wait.

</div>

## Learning objectives

In today's session we will:

* Try out the *JAX* differentiable programming framework.
* Learn about *forward* and *reverse* modes applied to functions of several variables.
* See showcases of more advanced AD usage.
* Learn about *checkpointing* and using AD to compute *higher order derivatives*.

## Preparations

#### Terminology

This course introduces the concept of *differentiable programming*, a.k.a. *automatic differentiation (AD)*, or *algorithmic differentiation*. We will use the acronym AD henceforth.

#### Notation

For a differentiable *mathematical* function $f:A\rightarrow\mathbb{R}$ with scalar input (i.e., a single value) from $A\subseteq\mathbb{R}$, we make use of both the Lagrange notation $f'(x)$ and Leibniz notation $\frac{\mathrm{d}f}{\mathrm{d}x}$ for its derivative.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
<b>Caution</b> with the physics notation for derivatives $\dot{x}$. It won't always mean what you expect! (See later.)
</div>

Similarly, for $m\in\mathbb{N}$ dimensional, differentiable, vector-valued function $\mathbf{f}:A\rightarrow\mathbb{R}^m$ with scalar input, we have derivative notations $\frac{\mathrm{d}\mathbf{f}}{\mathrm{d}x}$.

For a differentiable function with vector input (i.e., multiple inputs), we use partial derivative notation. For example, if $f:\mathbb{R}^2\rightarrow\mathbb{R}$ is written as $f=f(x,y)$ then we have the partial derivatives $\frac{\partial f}{\partial x}$ and $\frac{\partial f}{\partial y}$ with respect to first and second components, respectively. We use
$$\nabla f=\left(\frac{\partial f}{\partial x_1},\dots,\frac{\partial f}{\partial x_m}\right)$$
to denote the vector of all such partial derivatives. Similarly for vector-valued functions with multiple inputs.

When it comes to derivatives in code, we use the `_d` notation (for "derivative" or "dot"), which is standard in the AD literature. Its meaning will be described in due course.

## Introduction to JAX

JAX is a differentiable programming framework written in Python but with its own Domain Specific Language (DSL).

Like Autograd, JAX overloads much of NumPy. To avoid confusion with standard NumPy and `autograd.numpy`, let's import `jax.numpy` as `jnp` rather than `np`. The main difference is that, unlike standard NumPy arrays, `jnp` arrays are *immutable*.

Like Autograd, the basic gradient computation function is called `grad` and can be imported from the main library directly.

In [ ]:
import jax.numpy as jnp

In [ ]:
from jax import grad
help(grad)

## Computing higher-order derivatives of a scalar function in JAX

Let's start out by computing derivatives of the hyperbolic tangent function,

$$\tanh(x)=\frac{1-e^{-2x}}{1+e^{-2x}}.$$

This can be implemented as the Python function:

In [ ]:
def tanh(x):
    return (1.0 - jnp.exp((-2 * x))) / (1.0 + jnp.exp(-(2 * x)))

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">

<b>Exercise</b>
    
Below is the same code for plotting this function over the range $[-7,7]$. Add code for printing its first four derivatives over the same range using JAX.

Hint: Combine calls to JAX's `grad` function with a call to `vmap` at the outer-most level, which automatically transforms functions into their 'batched' version, allowing you to compute gradients of each entry in an array.

<b>Solution</b>
    
<details>

```python
from jax import grad, vmap

x = np.linspace(-7, 7, 700)

fig, axes = plt.subplots()
axes.plot(x, tanh(x), label=r"$\tanh(x)$")
axes.plot(x, vmap(grad(tanh))(x), label=r"$\tanh'(x)$")
axes.plot(x, vmap(grad(grad(tanh)))(x), label=r"$\tanh''(x)$")
axes.plot(x, vmap(grad(grad(grad(tanh))))(x), label=r"$\tanh'''(x)$")
axes.plot(x, vmap(grad(grad(grad(grad(tanh)))))(x), label=r"$\tanh''''(x)$")
axes.set_xlabel(r"$x$")
axes.set_ylabel(r"$y$")
axes.set_xlim([-7, 7])
axes.grid()
axes.legend();
```

</details>

</div>

In [ ]:
from jax import grad, vmap

x = np.linspace(-7, 7, 700)

fig, axes = plt.subplots()
axes.plot(x, tanh(x), label=r"$\tanh(x)$")
# TODO: Plot first-, second-, third-, and fourth-order derivatives, too
axes.set_xlabel(r"$x$")
axes.set_ylabel(r"$y$")
axes.set_xlim([-7, 7])
axes.grid()
axes.legend();

## Recall: Forward and reverse mode for scalar functions

In session 1, given three scalar functions $f$, $g$, and $h$, which may be composed as
$$\ell(x)=h(g(f(x)),$$
we presented forward mode as evaluating from right-to-left:
$$\frac{\mathrm{d}\ell}{\mathrm{d}x}=\frac{\mathrm{d}h}{\mathrm{d}g}\left(\frac{\mathrm{d}g}{\mathrm{d}f}\frac{\mathrm{d}f}{\mathrm{d}x}\right)$$
and reverse mode as evaluating from left-to-right:
$$\frac{\mathrm{d}\ell}{\mathrm{d}x}=\left(\frac{\mathrm{d}h}{\mathrm{d}g}\frac{\mathrm{d}g}{\mathrm{d}f}\right)\frac{\mathrm{d}f}{\mathrm{d}x}.$$

## Forward mode for vector functions

Suppose we have a function mapping between vectors, $\mathbf{f}:\mathbb{R}^m\rightarrow\mathbb{R}^n$ and a point $\mathbf{x}\in\mathbb{R}^m$ at which we seek to evaluate its derivative.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
<b>Note</b>
We can interpret such a function as having one or more scalar inputs and one or more scalar outputs.
</div>

For a general definition of forward mode, we need to consider a *seed vector*, $\dot{\mathbf{x}}\in\mathbb{R}^m$. Forward mode allows us to compute the *action* (matrix-vector product)
$$\text{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}}):=\nabla\mathbf{f}(\mathbf{x})\,\dot{\mathbf{x}}.$$
Here $\nabla\mathbf{f}$ is referred to as the *Jacobian* for the map, so the above is known as a *Jacobian-vector product (JVP)*.
You might also hear the related term *tangent linear model (TLM)*.

Think of the seed vector as the direction in which we want to compute the derivative.
For example, using $\dot{\mathbf{x}} = (1, 0, 0, ...)$ would give the derivative with respect to the first scalar input.
In practice, the seed vector is often a derivative of some upstream code from outside of the part of the program being differentiated.
That is, the upstream code is *passive*, whereas the part we are interested in is *active*, as far as AD is concerned.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
<b>Note</b>
The computation is <em>matrix-free</em>. We don't actually need to assemble the Jacobian when we compute this product.
</div>

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">
<b>Question</b>
    
In the scalar case we assumed a value for the seed. What was it?

</div>

## $f$ \& $g$ example

Consider two functions acting on real numbers:
$$f(x,y)=xy$$
and
$$g(z)=(\sin(z),\cos(z)).$$
Here $f:\mathbb{R}^2\rightarrow\mathbb{R}$ takes two inputs and returns a single output, while $g:\mathbb{R}\rightarrow\mathbb{R}^2$ takes a single input and returns two outputs.

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">
    
<b>Optional exercise</b>

Convince yourself that it is well defined for these functions may be composed in either order. (Although they won't necessarily give the same value!)

<b>Solution</b>

<details>
        
The image of $f$ is the set of all real numbers, so its image is the same as the domain of $g$ (i.e., $\text{im}(f)=\mathbb{R}=\text{dom}(g)$).

The image of $g$ is $[-1,1]^2=[-1,1]\times[-1,1]$ because $\sin$ and $\cos$ give values between -1 and 1. Since this is a subset of $\mathbb{R}^2$, the image of $g$ is a subset of the domain of $f$ (i.e., $\text{im}(g)\subset\text{dom}(f)$).

</details>
</div>

## $f$ \& $g$ example: Directed Acyclic Graph

We can visualise the functions in terms of DAGs.

Recalling that
$$f(x_1,x_2)=x_1x_2$$
and
$$g(y)=(\sin(y),\cos(y))\, ,$$
we have

<div style="text-align: center;">
  <img src="images/f_dag.png" width="400" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 1:</strong> Directed Acyclic Graph (DAG) for the $f$ function in the $f$ & $g$ example. Generated using tikZ and $\LaTeX$.
</div>

<div style="text-align: center;">
  <img src="images/g_dag.png" width="400" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 2:</strong> Directed Acyclic Graph (DAG) for the $g$ function in the $f$ & $g$ example. Generated using tikZ and $\LaTeX$.
</div>

In [ ]:
def f(x):
    return jnp.asarray([x[0] * x[1]])

In [ ]:
def g(y):
    return jnp.asarray([jnp.sin(y), jnp.cos(y)])

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Demonstrate JVP using JAX for the f \& g problem

</div>

In [ ]:
from jax import jacfwd

In [ ]:
x = jnp.array([2.0, 0.0])
print(jacfwd(f)(x)

## Chaining forward mode derivatives

But how does this correspond to the case with three functions?

Suppose in addition to $\mathbf{f}:\mathbb{R}^m\rightarrow\mathbb{R}^n$ we have $\mathbf{g}:\mathbb{R}^n\rightarrow\mathbb{R}^k$, $\mathbf{h}:\mathbb{R}^k\rightarrow\mathbb{R}^r$, and their composition $\boldsymbol{\ell}=\mathbf{f}\circ\mathbf{g}\circ\mathbf{h}$, where the notation here implies
$$
\mathbf{f}\circ\mathbf{g}\circ\mathbf{h}(\mathbf{x})=\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x})))
$$
for $\mathbf{x}\in\mathbb{R}^m$. Then by two applications of the chain rule
$$
\begin{align}
\mathrm{JVP}(\boldsymbol{\ell},\mathbf{x},\dot{\mathbf{x}})
&=\mathrm{JVP}(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h},\,\mathbf{x},\,\dot{\mathbf{x}})\\
&=\nabla(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h})(\mathbf{x})\,\dot{\mathbf{x}}\\
&=\nabla(\mathbf{g}\circ\mathbf{h})(\mathbf{f}(\mathbf{x}))\,\left(\nabla\mathbf{f}(\mathbf{x})\,\dot{\mathbf{x}}\right)\\
&=\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\left(\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\left(\nabla\mathbf{f}(\mathbf{x})\,\dot{\mathbf{x}}\right)\right)\\
&=\mathrm{JVP}(\mathbf{h},\,\mathbf{g}(\mathbf{f}(\mathbf{x})),\,\mathrm{JVP}(\mathbf{g},\,\mathbf{f}(\mathbf{x}),\,\mathrm{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}})))
\end{align}
$$
or can be written as
$$
\begin{align}
\mathrm{JVP}(\boldsymbol{\ell},\mathbf{x},\dot{\mathbf{x}})
&=\mathrm{JVP}(\mathbf{h},\,\mathbf{g}(\mathbf{f}(\mathbf{x})),\,\mathbf{J}_{\mathbf{g}(\mathbf{f}(\mathbf{x}))})\\
\mathbf{J}_{\mathbf{g}(\mathbf{f}(\mathbf{x}))}
&=\mathrm{JVP}(\mathbf{g},\,\mathbf{f}(\mathbf{x}),\,\mathbf{J}_{\mathbf{f}(\mathbf{x})})\\
\mathbf{J}_{\mathbf{f}(\mathbf{x})}
&=\mathrm{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}})
\end{align}
$$
So if we can compute a function and evaluate its forward mode derivative at the same time then we can compute such compositions and their derivatives efficiently, too.

## $f$ \& $g$ example: composition

Consider the composition $h=f\circ g:\mathbb{R}^2\rightarrow\mathbb{R}^2$, which is given by
$$h(x_1,x_2)=(f\circ g)(x_1,x_2)=g(f(x_1,x_2))=g(x_1x_2)=(\sin(x_1x_2),\cos(x_1x_2)).$$

For the derivative of each component,
$$
\frac{\partial f}{\partial x_1}=\frac{\partial}{\partial x_1}x_1x_2=x_2,
\quad\frac{\partial f}{\partial x_2}=\frac{\partial}{\partial x_2}x_1x_2=x_1,
\quad\frac{\partial g}{\partial y}=\frac{\partial}{\partial y}(\sin(y),\cos(y))=(\cos(y),-\sin(y)).
$$

Introduce the notation $g(y)=(g_1(y),g_2(y))$ so that
$$
\frac{\partial g_1}{\partial y}=\cos(y),
\quad\frac{\partial g_2}{\partial y}=-\sin(y).
$$
Similarly $h(x_1,x_2)=(h_1(x_1,x_2),h_2(x_1,x_2))=(g_1(f(x_1,x_2)),g_2(f(x_1,x_2)))$.

Let's use the chain rule to work out the derivatives of each of the outputs with respect to each of the inputs.
$$
\frac{\partial h_1}{\partial x_1}=\frac{\partial g_1}{\partial f}\frac{\partial f}{\partial x_1}=\cos(y)x_2=x_2\cos(x_1x_2),
\quad\frac{\partial h_1}{\partial x_2}=\frac{\partial g_1}{\partial f}\frac{\partial f}{\partial x_2}=\cos(y)x_1=x_1\cos(x_1x_2),
$$
where $y=f(x_1,x_2)$ and
$$
\quad\frac{\partial h_2}{\partial x_1}=\frac{\partial g_2}{\partial f}\frac{\partial f}{\partial x_1}=-\sin(y)x_2=-x_2\sin(x_1x_2),
\quad\frac{\partial h_2}{\partial x_2}=\frac{\partial g_2}{\partial f}\frac{\partial f}{\partial x_2}=-\sin(y)x_1=-x_1\sin(x_1x_2).
$$

We will come back to these formulae to verify the correctness of our AD computations.

## $f$ \& $g$ example: Seed vectors

Let's revisit the DAG interpretation and consider how the derivatives work.

<div style="text-align: center;">
  <img src="images/forward_dag.png" width="400" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 3:</strong> Directed Acyclic Graph (DAG) for the composition of the functions in the $f$ & $g$ example.
</div>

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Demonstrate chaining JVP using JAX for the f \& g problem.

</div>

## Reverse mode for vector functions

Consider the same vector function as above, $\mathbf{f}:\mathbb{R}^m\rightarrow\mathbb{R}^n$ and a point $\mathbf{x}\in\mathbb{R}^m$.

Given $\mathbf{x}\in\mathbb{R}^m$ and a seed vector $\bar{\mathbf{y}}\in\mathbb{R}^n$, reverse mode AD allows us to compute the *transpose action* (transposed matrix-vector product)
$$\text{JTVP}(\mathbf{f},\mathbf{x},\bar{\mathbf{y}}):=\nabla\mathbf{f}(\mathbf{x})^T\bar{\mathbf{y}}.$$

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Notes:</b>

- The dimension of the seed vector corresponds to that of the output rather than the input.
- Again, the computation is <em>matrix-free</em>. We don't actually need the Jacobian or its transpose when we compute this product.
- Our original definition for the scalar case assumed the same seed as above.

</div>

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">

<b>Optional exercises</b>

1. Convince yourself that the JTVP is well defined.
2. Repeat the above exercise to convice yourself that the definition coincides with the scalar definition, i.e.

$$
\mathrm{JTVP}(\boldsymbol{\ell},\mathbf{x},\bar{\mathbf{y}})
=\mathrm{JTVP}(\mathbf{f},\,\mathbf{x},\,\mathrm{JTVP}(\mathbf{g},\,\mathbf{f}(\mathbf{x}),\,\mathrm{JTVP}(\mathbf{h},\mathbf{g}(\mathbf{f}(\mathbf{x})),\bar{\mathbf{y}}))).
$$

<b>Solution 1</b>

<details>

We have $\nabla\mathbf{f}(\mathbf{x})\in\mathbb{R}^{n\times m}$, so $\nabla\mathbf{f}(\mathbf{x})^T\in\mathbb{R}^{m\times n}$. Since $\bar{\mathbf{y}}\in\mathbb{R}^n$, the dimensions are appropriate to take the JTVP.

</details>

<br>
<b>Solution 2</b>

<details>

$$
\begin{align}
\mathrm{JTVP}(\boldsymbol{\ell},\mathbf{x},\bar{\mathbf{y}})
&=\mathrm{JVP}(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h},\,\mathbf{x},\,\bar{\mathbf{y}})\\
&=\nabla(\mathbf{f}\circ\mathbf{g}\circ\mathbf{h})(\mathbf{x})^T\,\bar{\mathbf{y}}\\
&=\left(\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))\,\nabla\mathbf{f}(\mathbf{x})\right)^T\bar{\mathbf{y}}\\
&=\left(\nabla\mathbf{f}(\mathbf{x})^T\,\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\,\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\right)\,\bar{\mathbf{y}}\\
&=\nabla\mathbf{f}(\mathbf{x})^T\,\left(\nabla\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\,\left(\nabla\mathbf{h}(\mathbf{g}(\mathbf{f}(\mathbf{x}))^T\,\bar{\mathbf{y}}\right)\right)\\
&=\mathrm{JTVP}(\mathbf{f},\,\mathbf{x},\,\mathrm{JTVP}(\mathbf{g},\,\mathbf{f}(\mathbf{x}),\,\mathrm{JTVP}(\mathbf{h},\mathbf{g}(\mathbf{f}(\mathbf{x})),\bar{\mathbf{y}})))
\end{align}
$$

where we have used the result from before and applied the rule of transposes for matrix multiplication.

</details>


</div>

## $f$ \& $g$ example: reverse mode seed vectors

In reverse mode, gradient information propagates in the opposite direction.

<div style="text-align: center;">
  <img src="images/reverse_dag.png" width="400" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 4:</strong> Directed Acyclic Graph (DAG) for the composition of the functions in the $f$ & $g$ example. Generated using tikZ and $\LaTeX$.
</div>

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Demonstrate JTVP using JAX for the f \& g problem.

</div>

## Forward mode vs. reverse mode

For seed vectors $\dot{\mathbf{x}}\in\mathbb{R}^n$ and $\bar{\mathbf{y}}\in\mathbb{R}^m$, forward mode and reverse mode compute

$$
    \text{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}}):=\nabla\mathbf{f}(\mathbf{x})\dot{\mathbf{x}}
    \quad\text{and}\quad
    \text{JTVP}(\mathbf{f},\mathbf{x},\bar{\mathbf{y}}):=\nabla\mathbf{f}(\mathbf{x})^T\bar{\mathbf{y}}
$$
respectively.

* Forward mode is more appropriate if $n\ll m$, i.e., $\#inputs\ll\#outputs$.
  * e.g., sensitivity analysis or optimisation w.r.t. a small number of parameters.
* Reverse mode is more appropriate if $n\gg m$, i.e., $\#inputs\gg\#outputs$.
  * e.g., ODE/PDE-constrained optimisation (cost function), machine learning training (loss function), goal-oriented error estimation (quantity of interest).
* Forward mode is computed *eagerly*, whereas reverse mode is done separately from the primal run.
* Reverse mode tends to have higher memory requirements.

## Computing Hessians in the vector case

For seed vectors $\dot{\mathbf{x}}\in\mathbb{R}^n$ and $\bar{\mathbf{y}}\in\mathbb{R}^m$, forward mode and reverse mode compute

$$
    \text{JVP}(\mathbf{f},\mathbf{x},\dot{\mathbf{x}}):=\nabla\mathbf{f}(\mathbf{x})\dot{\mathbf{x}}
    \quad\text{and}\quad
    \text{JTVP}(\mathbf{f},\mathbf{x},\bar{\mathbf{y}}):=\nabla\mathbf{f}(\mathbf{x})^T\bar{\mathbf{y}}
$$
respectively.

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">

<b>Question</b>
    
What are two ways we can we use these to compute the Hessian of $f$?

<b>Solution 1</b>
    
<details>

Given a seed vector $\dot{\mathbf{x}}$, first apply forward mode to compute $\nabla\mathbf{f}(\mathbf{x})\dot{\mathbf{x}}$. Then apply forward mode to compute the gradient of *this* (i.e., apply forward mode to the *forward mode derivative code*). Use vector mode (preferably with compression!) to get the full Hessian.

</details>

<br>
<b>Solution 2</b>
    
<details>

Given a seed vector $\dot{\mathbf{x}}$, first apply forward mode to compute $\nabla\mathbf{f}(\mathbf{x})\dot{\mathbf{x}}$. Then apply reverse mode to compute the gradient of *this* (i.e., apply reverse mode to the *forward mode derivative code*). That is, $(\nabla(\nabla\mathbf{f}(\mathbf{x})\dot{\mathbf{x}}))^T\bar{\mathbf{y}}=\dot{\mathbf{x}}^T\nabla^T\nabla\mathbf{f}(\mathbf{x})\bar{\mathbf{y}}$. Here the Hessian $\mathbf{H}(\mathbf{f}):=\nabla^T\nabla\mathbf{f}(\mathbf{x})$ is symmetric and so the two applications give the Hessian-vector product with the seed. Use vector mode (preferably with compression!) to get the full Hessian.

</details>

</div>

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Demonstrate Hessian computation using JAX.

</div>

## Source transformation

In this course, we consider the operator overloading strategy for implementing AD. An alternative approach is to use *source transformation*.

High level idea: Given some (code) function `f(x)` and a seed vector `x_d`, generate the code for the function `f_d(x, x_d)` for its (directional) derivative.

This can be viewed as 'static analysis', which needs to be done ahead of (compiling and) running your test case. Often the difficult part is hooking the differentiated code into the wider model/build system.

#### Brief example

In Session 1 of last year's Differentiable Programming course (see [recording on YouTube](https://www.youtube.com/watch?v=R8b-vTMvN80)), we considered the Tapenade source transformation tool applied to Fortran code.

Applying Tapenade to the following subroutine for computing the product of two scalar arguments
```fortran
subroutine f(x, y)
  implicit none
  real, dimension(2), intent(in)  :: x
  real, intent(out) :: y
  y = x(1) * x(2)
end subroutine f
```
generates code
```fortran
!        Generated by TAPENADE     (INRIA, Ecuador team)
!  Tapenade 3.16 (develop) - 23 Apr 2025 13:39
!
!  Differentiation of f in forward (tangent) mode:
!   variations   of useful results: y
!   with respect to varying inputs: x
!   RW status of diff variables: x:in y:out
SUBROUTINE F_D(x, xd, y, yd)
  IMPLICIT NONE
  REAL, DIMENSION(2), INTENT(IN) :: x
  REAL, DIMENSION(2), INTENT(IN) :: xd
  REAL, INTENT(OUT) :: y
  REAL, INTENT(OUT) :: yd
  yd = x(2)*xd(1) + x(1)*xd(2)
  y = x(1)*x(2)
END SUBROUTINE F_D
```

## Source transformation vs. operator overloading

We already evaluated the differences between modes (forward and reverse). How about the differences between approaches?

* ST is done as a preprocessing step, whereas OO is done at runtime.
* ST is fairly clear, whereas OO is somewhat of a 'black box' (unless you're able to inspect the tape).
* OO's tape requires memory.
* While there are many OO tools (see Session 1), there are only a few ST tools:
  * [Tapenade](https://tapenade.gitlabpages.inria.fr/userdoc/build/html/index.html) (C, Fortran, Julia*)
  * [OpenAD](https://www.mcs.anl.gov/OpenAD) (Fortran) [no longer maintained]
  * [TAF](http://fastopt.com/products/taf) (Fortran) [commercial]
  * [PSyAD](https://psyclone-adjoint.readthedocs.io/en/stable)* (domain-specific)

*\*Work in progress*

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: More interesting JAX example ([#24](https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2026/issues/24))

</div>

## Checkpointing

Recall our discussion about tape unrolling and how for reverse mode we first unroll the tape to run the primal code.

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">

<b>Question</b>

Under what circumstances can we get away without an initial forward pass when running reverse mode?

<b>Solution</b>

<details>

Two possible answers:

If the variables involved in the forward computation had their values assigned exactly once and those values are still available in memory then we already have the forward data required for the reverse pass.*

If the problem is linear in the independent variables then the reverse mode code will be *independent of* those variables. As such, there is no need to compute them.

*Although your AD tool might not be aware of this or be able to make use of the information.

</details>

</div>

## Checkpointing

In scientific programming, time-dependent problems are typically solved using a timestepping method such as the theta-method we saw in the first session. When writing code for such methods, it's common practice to overwrite the variable for the approximation at a given timestep as we progress through the steps. In such a formulation, the value is no longer in memory and needs to be recomputed.

In some cases, it's possible to keep the full state on the tape at every timestep, whereby all the information required for reverse mode is available - see Figure 5.

<div style="text-align: center;">
  <img src="images/in_memory.png" width="600" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 5:</strong> Time-dependent adjoint problem solved without loading checkpoints. Generated using tikZ and $\LaTeX$.
</div>

<br>

This quickly becomes infeasible for real world problems, with the amount of memory required at each timestep taking a large chunk of the available RAM. In the extreme case where only one timestep's worth of data fits in memory, a reverse mode propagation requires checkpointing at each timestep as in Figure 6.

<div style="text-align: center;">
  <img src="images/load_all.png" width="800" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 6:</strong> Time-dependent adjoint problem solved with checkpointing at every timestep. Generated using tikZ and $\LaTeX$.
</div>

<br>

In intermediate cases, checkpointing can be done on the basis of some fixed frequency or in a more intelligent way using the *revolve* algorithm [(Griewank & Walther, 2000)](https://doi.org/10.1145/347837.347846).

## Further applications

* Sensitivity analysis. <!-- Compute gradients of outputs with respect to parameters of interest -->
* Data assimilation. <!-- Uses gradients of cost functions involving mismatches against observations to assimilate the observations. -->
* Uncertainty quantification. <!-- Uses Hessians -->
* Online training in machine learning. <!-- Requires derivatives of code downstream from machine learning outputs -->
* PDE-constrained optimisation.  <!-- Typically uses gradient-based methods -->
* Goal-oriented error estimation and mesh adaptation. <!-- Uses adjoint solutions -->

<div class="alert alert-block alert-danger"; border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
TODO: Pitch mini-project ([#23](https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2026/issues/23))

</div>

## Summary and outlook

In today’s session we:

* Tried out the JAX differentiable programming framework.
* Learnt about forward and reverse modes applied to functions of several variables.
* Saw showcases of more advanced AD usage.
* Learnt about checkpointing and using AD to compute higher order derivatives.

## European workshop on Automatic differentiation

<div style="text-align: center;">
  <img src="https://cambridge-iccs.github.io/euroad29/_images/cms.png" width="600" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 1:</strong> Centre of Mathematical Sciences, University of Cambridge</a>.
</div>
<br>
ICCS will be hosting the 29th European workshop on Automatic Differentiation (EuroAD) in Cambridge on the **29th-30th September 2026**. It will be an informal meeting of researchers and software engineers who develop and apply automatic differentiation theory and software.

Attendees from all career stages are welcome, particularly PhD students and early career researchers and software engineers.

We welcome contributions with both theoretical and practical perspectives. A selection of possible topics include: new methods for AD, software developments and inter-comparison, AD in machine learning, and applications in science, engineering, and beyond.

Register at
https://cambridge-iccs.github.io/euroad29/

## References

* [Autograd README](https://github.com/HIPS/autograd/blob/master/README.md)
* [Autograd Tutorial](https://github.com/HIPS/autograd/blob/master/docs/tutorial.md)
* [JAX documentation](https://docs.jax.dev)
* R. E. Wengert. *A simple automatic derivative evaluation program* (1964). Communications
of the ACM, 7(8):463–464, [doi.org:10.1145/355586.364791](https://doi.org/10.1145/355586.364791).
* S. Linnainmaa. *Taylor expansion of the accumulated rounding error*. BIT,
16(2):146–160, 1976, [doi:10.1007/BF01931367](https://doi.org/10.1007/BF01931367).
* B. Speelpenning. *Compiling fast partial derivatives of functions given by algorithms*.
University of Illinois, 1980, [doi:10.2172/5254402](https://doi.org/10.2172/5254402).
* A. Griewank. *Achieving logarithmic growth of temporal and spatial complexity in
reverse automatic differentiation.* Optimization Methods & Software, 1:35–54, 1992, [doi:10.1080/10556789208805505](https://doi.org/10.1080/10556789208805505).